In [ ]:

import torch
import pyro
import pyro.distributions as dist
import pyro.contrib.gp as gp
from pyro.infer import MCMC, NUTS, HMC
from IPython.core.pylabtools import figsize
import numpy as np
import pandas as pd

from pathlib import Path
import glob
from matplotlib import pyplot as plt
from scipy.stats.mstats import mquantiles
from sklearn.preprocessing import MinMaxScaler
from pyro.infer.autoguide import init_to_value



# Parameter determination for function
We specify a bi-Lorentzian model for the ODMR spectra where the mean values change is driven by a learned linear function that depends on temperature. In the first part, we learn the parameters of the linear "calibration" function that maps the peak center positions on to temperature. From previous experiment we know that the model is underspecified. Here we add an additional affine transfomation on the output of the probablistic model. This learned model with calibration added is referred to as the calibrated_model-  it is then subsequently used to predict temperatures given spectra and associated uncertainty in the predicted value.

## Set up function
Here we import the data, define a bi-Lorentzian function and demonstrate the impact of different values of center 1 and center 2 (A and B, respectively) on the simulated spectra.

In [ ]:
temps_ = [-30, 15, 25, 30, 40, 70, 40, 25, 10, -30]

temps = np.array(temps_, dtype=float)

temps = temps + 273.15  # Convert to Kelvin

temps = torch.tensor(temps).double()

temps = temps[:6]

temp_test_array = np.array(temps_, dtype=float) + 273.15  # Convert to Kelvin
temp_test = torch.tensor(temp_test_array).double()

In [ ]:
print(len(temps))
temps-273.15

In [ ]:
## import data file
fpath = '../saved_data/cycle3'

df_= pd.read_csv(fpath, sep=',', header = 0); 
df = df_.iloc[:, 1:]
df.iloc[:, 2:].plot(legend= False)

# define and scale the frequency axis 
x_esr = df.frequency.values
y_esr = df.iloc[:, 1:7]

#scale x axis to 0-100
sc = MinMaxScaler(feature_range=(0, 100))
x_scale = sc.fit_transform(x_esr.reshape(-1, 1)).flatten()

torch.set_default_dtype(torch.float64)


y_esr_ = df.iloc[:, 1:]



In [ ]:
df.shape
print(y_esr.columns)
print(temps)

In [ ]:
# plot baseline subtracted  y data
y_esr = y_esr.apply(lambda x: x - x[:20].mean())#+0.01
y_esr = -1*y_esr
y_esr = y_esr.apply(lambda x: x/x.max())
plt.plot(x_scale, y_esr); plt.show()
#plt.xlim(40, 80)

y_esr_ = y_esr_.apply(lambda x: x - x[:20].mean())#+0.01
y_esr_ = -1*y_esr_
y_esr_ = y_esr_.apply(lambda x: x/x.max())
plt.plot(x_scale, y_esr_);

In [ ]:
#numpy version of function
def F_np(x_in, Amp1, Amp2, alpha, beta, X, gamma1, gamma2, T):
    # Reshape A, B, Amp, G1, G2 for broadcasting with x_in
    alpha = alpha[None, :]
    Amp1_reshaped = Amp1[None, :]
    Amp2_reshaped = Amp2[None, :]
    T_reshaped = T[None, :]
    beta = beta[None, :]
    A_reshaped = (T_reshaped*alpha) + beta # Shape (1, num_samples)
    X_reshaped = X[None, :]
    B_reshaped = A_reshaped + X_reshaped
     # Shape (1, num_samples)
    G1_reshaped = gamma1[None,: ] # New: Reshape gamma1
    G2_reshaped = gamma2[None, :] # New: Reshape gamma2
    
    x_in_reshaped = x_in[:, None] # Shape (num_x_points, 1)

    F = (Amp1_reshaped) * (0.5 * G1_reshaped) / ((x_in_reshaped - A_reshaped)**2 + (0.5 * G1_reshaped)**2) \
        + (Amp2_reshaped) * (0.5 * G2_reshaped) / ((x_in_reshaped - B_reshaped)**2 + (0.5 * G2_reshaped)**2)
    return F


# A function to sample values of A and B and plot resulting function.
def F_samples():
    amp1 = pyro.sample("Amp1", dist.Normal(3.0, 0.25))
    amp2 = pyro.sample("Amp2", dist.Normal(0.95, 0.25))
    alpha = pyro.sample("alpha", dist.Normal(-71, 7.0)).double() /1000
    beta = pyro.sample("beta", dist.Normal(42., 10.0)) # Sample gamma1
    T_ = torch.tensor(273.15).double()
    T = T_ * np.ones_like(alpha) 
    A =  T*alpha + beta
    X = pyro.sample('X', dist.Normal(8.42, 0.05))
    B =  A + X
    gamma1 =  pyro.sample("gamma1", dist.Normal(7.6, 0.5)).double()
    gamma2 = pyro.sample("gamma2", dist.Normal(7.6, 0.5)).double()
    # Use the sampled gamma values in the function
    f_sim = lambda xi: (amp1) *(0.5*gamma1)/((xi-A)**2 + (0.5*gamma1)**2) + (amp2) *(0.5*gamma2)/((xi-B)**2 + (0.5*gamma2)**2)
    plt.plot(x_scale, f_sim(x_scale))
    # Update title to show sampled gamma values
    plt.title('A:' + str(A.numpy()) + ' B:' + str(B.numpy()) + ' G1:' + str(gamma1.numpy()) + ' G2:' + str(gamma2.numpy()))



## Define data we will start with and plot.

In [ ]:

# this will be the data we'll start with, shown as red circles
def dataslicer(x, y, col1 =0 ,col2=1):
    x_scale_tensor = torch.tensor(x).double()
    # squeeze the selected column to produce a 1-D tensor (N,) instead of (N,1)
    y_vals = y.iloc[:, col1:col2].values.squeeze()
    y_scale_tensor = torch.tensor(y_vals).double()
    return x_scale_tensor, y_scale_tensor



In [ ]:
plt.plot(x_scale, y_esr.iloc[:, 1].values, 'o')
plt.xlim(45,65)

## Set up and run Bayesian inference

In [ ]:
# define the model
# Here we assume that we know the noise variance in the data = 0.1,
# though this can be set to another parameter to learn.

x_obs, y_obs = dataslicer(x_scale, y_esr, col1=0, col2=1)
T_ = 273.15 + 25.0  # Example temperature in Kelvin

data_ = (T_, (x_obs.clone().detach().double(), y_obs.clone().detach().double()))

def model(data_):
    T, data, = data_[0], data_[1]
    alpha = pyro.sample("alpha", dist.Normal(-0.077, 0.01)).double() 
    beta = pyro.sample("beta", dist.Normal(70., 10.0)) # Sample gamma1
    var = pyro.sample("var", dist.HalfNormal(scale=0.1)).double()
    gamma1 =  pyro.sample("gamma1", dist.Normal(8.02, 1.0)).double()
    gamma2 =  pyro.sample("gamma2", dist.Normal(8.02, 1.0)).double()
    A =  T*alpha + beta
    X = pyro.sample('X', dist.Normal(8.42, 0.05))
    B =  A + X
    #gamma1 =  8.020510711744828 
    amp1 =  pyro.sample("amp1", dist.Normal(3., 0.25 )).double()
    amp2 =  pyro.sample("amp2", dist.Normal(3., 0.25 )).double()
    
    F_ =  (amp1) * (0.5 * gamma1) / ((data[0] - A)**2 + (0.5 * gamma1)**2) \
        + (amp2) * (0.5 * gamma2) / ((data[0] - B)**2 + (0.5 * gamma2)**2)

    # Make sure F_ is 1-D and compute residuals = observed - deterministic
    F = F_.squeeze()
    
       #with pyro.plate("data", data[0].size(0)):
    pyro.sample("obs", dist.MultivariateNormal(F, var * torch.eye(data[1].shape[0]).double()), obs=data[1])

In [ ]:
# solve for the posterior using MCMC
# Use the observed x positions and y values as the data passed to the model


init_vals = {
    "alpha": torch.tensor(-0.077),
    "beta": torch.tensor(71.0),
    "X": torch.tensor(8.4),
    "gamma1": torch.tensor(7.8),
    "gamma2":torch.tensor(7.8),
    "amp1": torch.tensor(3.0),
    "amp2": torch.tensor(3.0),
    "var": torch.tensor(0.05),}


kernel = NUTS(model, jit_compile=True, init_strategy=init_to_value(values=init_vals), max_tree_depth=6, ignore_jit_warnings=True)
posterior = MCMC(kernel, num_samples=200, warmup_steps=200, num_chains=1)
posterior.run((data_))



In [ ]:


hmc_samples = {k: v.detach().cpu().numpy() for k, v in posterior.get_samples().items()}
alpha = hmc_samples['alpha']
beta = hmc_samples['beta']
Amp1 = hmc_samples['amp1'] # Get posterior samples for amp
Amp2 = hmc_samples['amp2'] # Get posterior samples for amp
var = hmc_samples['var'] # Get posterior samples for var
gamma1 = hmc_samples['gamma1']
gamma2 = hmc_samples['gamma2']
X = hmc_samples["X"]
T = T_ * np.ones_like(alpha)  # Create T array matching alpha shape
F = F_np(x_scale, Amp1, Amp2, alpha, beta, X, gamma1, gamma2, T)
qs = mquantiles(F.T, [0.025, 0.975], axis=0)
F_mean = F.mean(axis = 1)

print('#################')
plt.fill_between(x_scale.flatten(), qs[0], qs[1], alpha=0.7, color="#7A68A6");
plt.plot(x_scale, F_mean)
plt.plot(x_obs, y_obs, 'ro'); # plotting the data for this slice
plt.xlabel('frequency axis')
plt.title('Posterior distribution for the function given distributions for all parameters');plt.show()
pyro.clear_param_store()
print('#################')


In [ ]:
print(f'alpha = {np.array(alpha).mean():.4f}')
print(f'alpha error = {np.array(alpha).var():.8f}')
print(f'beta = {np.array(beta).mean():.4f}')
print(f'beta error = {np.array(beta).var():.6f}')
print(f'X = {np.array(X).mean():.4f}')
print(f'X error = {np.array(X).var():.6f}')
print(f'gamma1 = {np.array(hmc_samples["gamma1"]).mean():.4f}')
print(f'gamma1 error = {np.array(hmc_samples["gamma1"]).var():.6f}')
print(f'gamma2 = {np.array(hmc_samples["gamma2"]).mean():.4f}')
print(f'gamma2 error = {np.array(hmc_samples["gamma2"]).var():.6f}')
print(f'amp1 = {np.array(hmc_samples["amp1"]).mean():.4f}')
print(f'amp1 error = {np.array(hmc_samples["amp1"]).var():.6f}')
print(f'amp2 = {np.array(hmc_samples["amp2"]).mean():.4f}')
print(f'amp2 error = {np.array(hmc_samples["amp2"]).var():.6f}')

## Get results and plot

In [ ]:
idx = []
error_alpha = []
error_beta = []
alpha_, beta_, amp_1, amp_2 = [], [], [], []
gamma1_vals, gamma2_vals = [], []
beta_var, alpha_var, gamma1_var, gamma2_var, amp_var1, amp_var2 = [], [], [], [], [], []
X_, X_vals = [], []

# init vals for MCMC
init_vals = {
    "alpha": torch.tensor(-0.076),
    "beta": torch.tensor(71.0),
    "X": torch.tensor(8.4),
    "gamma1": torch.tensor(7.8),
    "gamma2": torch.tensor(7.8),
    "amp": torch.tensor(3.0),
    "var": torch.tensor(0.05),}

for j in range(0, y_esr.shape[1]):
  print(j)
  print(init_vals)
  x_obs_j, y_obs_j = dataslicer(x_scale, y_esr, col1=j, col2=j+1)
  temps_j = temps[j]
  data_j_ = (temps_j, (x_obs_j[20:].clone().detach().double(), y_obs_j[20:].clone().detach().double()))
  kernel = NUTS(model, jit_compile=True, init_strategy=init_to_value(values=init_vals), ignore_jit_warnings=True, max_tree_depth=6)
  posterior = MCMC(kernel, num_samples=200, warmup_steps=200, num_chains=1)
  posterior.run(data_j_)
  hmc_samples = {k: v.detach().cpu().numpy() for k, v in posterior.get_samples().items()}
  alpha = hmc_samples['alpha']
  beta = hmc_samples['beta']
  Amp1 = hmc_samples['amp1'] # Get posterior samples for amp
  Amp2 = hmc_samples['amp2'] # Get posterior samples for amp
  var = hmc_samples['var'] # Get posterior samples for var
  X = hmc_samples["X"]
  T = temps_j.item() * np.ones_like(alpha)  # Create T array matching alpha shape
  gamma1_ = hmc_samples['gamma1'] # Get posterior samples for gamma1
  gamma2_ = hmc_samples['gamma2'] # Get posterior samples for gamma2 (since gamma2 = gamma1 in the model)
  var = hmc_samples['var'] # Get posterior samples for var
  X_.append(X.mean())
  X_vals.append(X.var())
  F = F_np(x_scale, Amp1, Amp2, alpha, beta, X, gamma1, gamma2, T)
  qs = mquantiles(F.T, [0.025, 0.975], axis=0)
  F_mean = F.mean(axis = 1)
  idx.append(j)
  amp_1.append(Amp1.mean())
  amp_2.append(Amp2.mean())  
  alpha_.append(alpha.mean())
  beta_.append(beta.mean())
  gamma1_vals.append(gamma1_.mean())
  gamma2_vals.append(gamma2_.mean())
  alpha_var.append(alpha.var())
  beta_var.append(beta.var())
  gamma1_var.append(gamma1_.var())
  gamma2_var.append(gamma2_.var())
  amp_var1.append(Amp1.var())
  amp_var2.append(Amp2.var())
  print('#################')
  plt.fill_between(x_scale.flatten(), qs[0], qs[1], alpha=0.7, color="#7A68A6")
  plt.plot(x_scale, F_mean)
  plt.plot(x_obs_j, y_obs_j, 'ro'); # plotting the data for this slice
  plt.xlabel('frequency axis')
  plt.title('Posterior distribution for the function given distributions for all parameters');plt.show()
  #pyro.clear_param_store()
  init_vals = {
      "alpha": torch.tensor(alpha.mean()),
      "beta": torch.tensor(beta.mean()),
      "X": torch.tensor(X.mean()),
      "gamma1": torch.tensor(gamma1_.mean()),
      "gamma2": torch.tensor(gamma2_.mean()),
      "amp1": torch.tensor(Amp1.mean()),
      "amp2": torch.tensor(Amp2.mean()),
      "var": torch.tensor(var.mean()),}    
  print('#################')


In [ ]:
print(f'alpha = {np.array(alpha_).mean():.4f}')
print(f'alpha error = {np.array(alpha_var).mean():.6f}')

print(f'beta = {np.array(beta_).mean():.4f}')
print(f'beta error = {np.array(beta_var).mean():.6f}')


print(f'gamma1 = {np.array(gamma1_).mean():.4f}')
print(f'gamma1 error = {np.array(gamma1_var).mean():.6f}')

print(f'gamma2 = {np.array(gamma2_).mean():.4f}')
print(f'gamma2 error = {np.array(gamma2_var).mean():.4f}')

In [ ]:
gamma1_vals = np.asarray(gamma1_vals)
gamma1_err = np.asarray(gamma1_var)
gamma1_high = gamma1_vals + gamma1_err
gamma1_low = gamma1_vals - gamma1_err

print(gamma1_high)
print(gamma1_low)

gamma_vals = np.asarray(gamma2_vals)
gamma2_err = np.asarray(gamma2_var)
gamma2_high = gamma2_vals + gamma2_err
gamma2_low = gamma2_vals - gamma2_err

In [ ]:
plt.plot(temps, gamma1_vals)
plt.fill_between(temps, gamma1_low,
    gamma1_high,
    color='blue',
    alpha=0.3,
    label='1σ uncertainty'
)
plt.plot(temps,gamma2_vals)
plt.fill_between(temps, gamma2_low,
    gamma2_high,
    color='orange',
    alpha=0.3,
    label='1σ uncertainty'
)
plt.xlabel('Temperature (K)')
plt.ylabel('Width (AU)')
plt.legend(['left peak', 'k = 1','right peak', 'k = 1'])

In [ ]:
amp_1_vals = np.asarray(amp_1)
amp1_err = np.asarray(amp_var1)
amp1_high = amp_1_vals + amp1_err
amp1_low = amp_1_vals - amp1_err

print(amp1_high)
print(amp1_low)

amp2_vals = np.asarray(amp_2)
amp2_err = np.asarray(amp_var2)
amp2_high = amp2_vals + amp2_err
amp2_low = amp2_vals - amp2_err

In [ ]:
plt.plot(temps, amp_1_vals)
plt.fill_between(temps, amp1_low,
    amp1_high,
    color='blue',
    alpha=0.3,
    label='1σ uncertainty'
)
plt.plot(temps, amp2_vals)
plt.fill_between(temps, amp2_low,
    amp2_high,
    color='orange',
    alpha=0.3,
    label='1σ uncertainty'
)
plt.xlabel('Temperature (K)')
plt.ylabel('amp (AU)')
plt.legend(['left peak', 'k = 1','right peak', 'k = 1'])

entire x range 
alpha = -0.0722
alpha error = 0.000000
beta = 70.6674
beta error = 0.012373
gamma1 = 7.7124
gamma1 error = 0.006962

In [ ]:
### define the calibrated model using learned alpha, beta and gamma parameters from the MCMC runs above, and use it to predict T for calibration data

def calibrated_model(data_):
    data = data_
    # Use learned calibration parameters (fixed from calibration phase)
    alpha = torch.tensor(np.array(alpha_).mean()).double()
    beta = torch.tensor(np.array(beta_).mean()).double()
    gamma1 = torch.tensor(np.array(gamma1_vals).mean()).double()  # pyro.sample("gamma1", dist.Normal(7.8, 0.2))
    gamma2 = torch.tensor(np.array(gamma2_vals).mean()).double()
    # Sample observation noise variance
    var = pyro.sample("var", dist.HalfNormal(scale=0.1)).double()
    # Sample amplitude with informative prior
    amp1 = torch.tensor(np.array(amp_1).mean()).double()  #pyro.sample("amp1", dist.Normal(5., 0.25)).double()
    amp2 = torch.tensor(np.array(amp_2).mean()).double() #pyro.sample("amp2", dist.Normal(5., 0.25)).double()
    # Sample temperature (prior from broad range, will be refined by data and GP)
    T_ = pyro.sample("T_", dist.Uniform(242, 344)).double()
    # T_ = T_ * np.ones_like(alpha)  # Create T array matching alpha shape
    A = T_ * alpha + beta
    X = torch.tensor(np.array(X_vals).mean()).double()
    B = A + X
    F_ = (amp1) * (0.5 * gamma1) / ((data[0] - A)**2 + (0.5 * gamma1)**2) \
        + (amp2) * (0.5 * gamma2) / ((data[0] - B)**2 + (0.5 * gamma2)**2)
    F_ = F_.squeeze()
    # Observe data
    pyro.sample("obs", dist.MultivariateNormal(F_, var * torch.eye(data[1].shape[0]).double()), obs=data[1])

In [ ]:
#numpy version of function
def F_np_calibrated(x_in, T):
    # Reshape A, B, Amp, G1, G2 for broadcasting with x_in
    alpha = np.array(alpha_).mean()
    Amp1_reshaped = np.array(amp_1).mean()
    Amp2_reshaped = np.array(amp_2).mean()
    T_reshaped = T[None, :]
    beta = np.array(beta_).mean()
    A_reshaped = (T_reshaped*alpha) + beta # Shape (1, num_samples)
    X_reshaped = np.array(X_).mean()
    B_reshaped = A_reshaped + X_reshaped
     # Shape (1, num_samples)
    G1_reshaped = np.array(gamma1).mean() # New: Reshape gamma1 gamma1[None, :]
    G2_reshaped = np.array(gamma2).mean() # New: Reshape gamma2

    x_in_reshaped = x_in[:, None] # Shape (num_x_points, 1)

    F = (Amp1_reshaped) * (0.5 * G1_reshaped) / ((x_in_reshaped - A_reshaped)**2 + (0.5 * G1_reshaped)**2) \
        + (Amp2_reshaped) * (0.5 * G2_reshaped) / ((x_in_reshaped - B_reshaped)**2 + (0.5 * G2_reshaped)**2)
    
    
    return F


In [ ]:
idx = []
temp_ = []
amp_1var, amp_2var = [], []
temp_var = []
temp_ci_high, temp_ci_low = [], []


# init vals for MCMC

init_vals = {
    
     "var": torch.tensor(0.05),}


for j in range(0, y_esr_.shape[1]):
  print(j)
  x_obs_j, y_obs_j = dataslicer(x_scale, y_esr_, col1=j, col2=j+1)
  temps_j = temp_test_array[j]
  data_j = (x_obs_j.clone().detach().double(), y_obs_j.clone().detach().double())
  # print(y_obs_j.shape)
  kernel = NUTS(calibrated_model, jit_compile=True, init_strategy=init_to_value(values=init_vals), ignore_jit_warnings=True, max_tree_depth=9)
  posterior = MCMC(kernel, num_samples=200, warmup_steps=1000, num_chains=1)
  posterior.run(data_j)
  hmc_samples = {k: v.detach().cpu().numpy() for k, v in posterior.get_samples().items()}
  T = hmc_samples['T_']
  var = hmc_samples['var'] # Get posterior samples for var
  F = F_np_calibrated(x_scale, amp1, amp2,  T)
  qs = mquantiles(F.T, [0.025, 0.975], axis=0)
  F_mean = F.mean(axis = 1)
  idx.append(j)
  temp_.append(T.mean())
  temp_var.append(T.var())
  amp_1var.append(Amp1.var())
  amp_2var.append(Amp2.var())
  T_ci = np.quantile(T, [0.025, 0.975])
  temp_ci_low.append(T_ci[0])
  temp_ci_high.append(T_ci[1])
  print('#################')
  print(f'experimental temperature: {temps_j.item():.2f} K, inferred temperature: {T.mean():.2f} K')
  plt.fill_between(x_scale.flatten(), qs[0], qs[1], alpha=0.7, color="#7A68A6");
  plt.plot(x_scale, F_mean)
  plt.plot(x_obs_j, y_obs_j, 'ro'); # plotting the data for this slice
  plt.xlabel('frequency axis')
  plt.title('Posterior distribution for the function given distributions for all parameters');plt.show()
  #pyro.clear_param_store()
 
  print('#################')

In [ ]:
print(f'amp = {np.array(amp_).mean():.4f}')
print(f'amp error = {np.array(amp_var).mean():.6f}')

print(f'gamma1 = {np.array(gamma1_vals).mean():.4f}')
print(f'gamma1 error = {np.array(gamma1_var).mean():.6f}')

print(f'Temperature = {np.array(temp_).mean():.2f} K')
print(f'Temperature error = {np.array(temp_var).mean():.6f}')

In [ ]:
temperature = np.asarray(temp_)
plt.plot(temperature, 'o-')
plt.plot(np.array(temp_test[:]), 'ro--')

In [ ]:
# plt.plot(np.array(temp_test[9:]), temperature[9:], 'k*')
# plt.xlabel('Experimental Temperature (K)')
# plt.ylabel('Inferred Temperature (K)')
# import seaborn as sns
# sns.regplot(x = np.array(temp_test[:9]), y = temperature[:9])

In [ ]:
np.sqrt(np.mean((np.asarray(temp_test[:len(temperature)]) -  temperature)**2))

In [ ]:
np.asarray(temp_test[:len(temperature)]) -  temperature

In [ ]:
plt.plot(temp_test_array[:6], np.asarray(temp_mean_corr[:6]), 'o')
plt.plot(temp_test_array[6:], np.asarray(temp_mean_corr[6:]), 'x')
plt.xlabel('Experimental Temperature (K)')
plt.ylabel('Inferred Temperature post correction (K)')
plt.legend(['Calibration data', 'Test data'])

mean_squared_error = np.mean((np.asarray(temp_test) -  np.asarray(temp_mean_corr))**2)
print(f'RMSE after linear regression correction: {np.sqrt(mean_squared_error):.2f} K')

In [ ]:
plt.plot(temp_test_array[:6], np.asarray(temp_mean_raw[:6]), 'o')
plt.plot(temp_test_array[6:], np.asarray(temp_mean_raw[6:]), 'x')
plt.xlabel('Experimental Temperature (K)')
plt.ylabel('Inferred Temperature post correction (K)')
plt.legend(['Calibration data', 'Test data'])

mean_squared_error = np.mean((np.asarray(temp_test) -  np.asarray(temp_mean_raw))**2)
print(f'RMSE after linear regression correction: {np.sqrt(mean_squared_error):.2f} K')

correction factor computed from up ramp jsut makes it worse

What is clear is that the there is significant hysteresis in this run that results in an offset error 